# 02 — Exploratory Data Analysis

Phase 2 of the methodology: establish the current global energy mix by technology and region before splitting into the three Phase 3 analysis modules (energy mix, demand gap, nuclear pipeline).

Sections:
1. Load processed data & post-ingestion sanity check
2. Global energy mix (current, operating capacity) — GIPT
3. Historical generation trend (1965-2024) — EI Statistical Review
4. GEM ↔ IEA region crosswalk check
5. Nuclear pipeline status breakdown — GNPT

In [1]:
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', None)

DATA_DIR = '../data/processed'

## 1. Load processed data & sanity check

Quick post-ingestion check: row counts, dtypes, and null rates should match what's documented in `CLAUDE.md`'s raw data schema notes. This isn't re-cleaning — just confirming ingestion didn't silently drop or mangle anything.

In [2]:
gipt = pd.read_parquet(f'{DATA_DIR}/gipt.parquet')
gnpt = pd.read_parquet(f'{DATA_DIR}/gnpt.parquet')
iea_world = pd.read_parquet(f'{DATA_DIR}/iea_world.parquet')
iea_regions = pd.read_parquet(f'{DATA_DIR}/iea_regions.parquet')
ei_stats = pd.read_parquet(f'{DATA_DIR}/ei_stats.parquet')

for name, df in [('gipt', gipt), ('gnpt', gnpt), ('iea_world', iea_world),
                  ('iea_regions', iea_regions), ('ei_stats', ei_stats)]:
    print(f'{name:12s} shape={df.shape}')

gipt         shape=(182428, 52)
gnpt         shape=(1749, 40)
iea_world    shape=(2316, 8)
iea_regions  shape=(2523, 8)
ei_stats     shape=(293234, 15)


In [3]:
# Null-rate spot check against CLAUDE.md schema notes
null_rates = pd.concat({
    'gipt': gipt.isna().mean(),
    'gnpt': gnpt.isna().mean(),
}, axis=1)
null_rates[(null_rates > 0).any(axis=1)].sort_values('gnpt', ascending=False)

,gipt,gnpt
retirement_date,NaN,0.878216
Planned Retirement,NaN,0.850200
Retirement Year,NaN,0.850200
Project Name in Local Language / Script,NaN,0.790738
construction_duration_yrs,NaN,0.769011
...,...,...
Captive Industry Use,0.971660,NaN
Captive Non Industry Use,0.981686,NaN
"Local area (taluk, county)",0.435591,NaN
"Major area (prefecture, district)",0.517695,NaN


In [4]:
print('GIPT status values:', sorted(gipt.status.dropna().unique()))
print('GIPT type values:', sorted(gipt.type.dropna().unique()))
print('GNPT status values:', sorted(gnpt.status.dropna().unique()))
print('IEA regions scenario coverage:')
print(iea_regions.scenario.value_counts())

GIPT status values: ['announced', 'cancelled', 'construction', 'mothballed', 'operating', 'pre-construction', 'retired', 'shelved']
GIPT type values: ['bioenergy', 'coal', 'geothermal', 'hydropower', 'nuclear', 'oil/gas', 'utility-scale solar', 'wind']
GNPT status values: ['announced', 'cancelled', 'construction', 'mothballed', 'operating', 'pre-construction', 'retired', 'shelved']
IEA regions scenario coverage:
scenario
Historical          1043
Current Policies     740
STEPS                740
Name: count, dtype: int64


Note: `iea_regions` has zero NZE rows post-ingestion (Current Policies / STEPS / Historical only) — confirms the CLAUDE.md flag that regional NZE analysis isn't feasible. `iea_world` retains all four scenarios including NZE.

## 2. Global energy mix (current) — GIPT

Operating capacity by technology `type`, globally and by region. This is the "what powers the world today" baseline.

In [5]:
operating = gipt[gipt.status == 'operating']

mix_global = (
    operating.groupby('type')['capacity_mw'].sum()
    .sort_values(ascending=False)
    .div(1000)  # GW
    .rename('capacity_gw')
)
mix_global

type
oil/gas                2232.94637
coal                   2202.52810
hydropower             1278.94000
utility-scale solar    1267.79890
wind                   1133.95360
nuclear                 401.34100
bioenergy               115.26461
geothermal               16.62676
Name: capacity_gw, dtype: float64

In [6]:
# Fix a single color per technology `type` up front. Without this, px assigns
# colors by first-appearance order in each chart's dataframe independently —
# since that order differs between the region- and subregion-level groupings,
# the same type (e.g. nuclear) can end up a different color in each chart.
TYPE_ORDER = mix_global.index.tolist()
TYPE_COLORS = dict(zip(TYPE_ORDER, px.colors.qualitative.Plotly))

In [7]:
fig = px.pie(
    mix_global.reset_index(), names='type', values='capacity_gw',
    title='Global operating capacity share by technology (GW)',
    color='type', color_discrete_map=TYPE_COLORS,
    category_orders={'type': TYPE_ORDER}
)
fig.show()

In [8]:
mix_by_region = (
    operating.groupby(['region', 'type'])['capacity_mw'].sum()
    .div(1000)
    .rename('capacity_gw')
    .reset_index()
)

fig = px.bar(
    mix_by_region, x='region', y='capacity_gw', color='type',
    title='Operating capacity by region and technology (GW)',
    barmode='stack', color_discrete_map=TYPE_COLORS,
    category_orders={'type': TYPE_ORDER}
)
fig.show()

In [9]:
# Same breakdown at the finer subregion grain — useful for spotting
# fast-growing developing regions (South/Southeast Asia, Sub-Saharan Africa)
mix_by_subregion = (
    operating.groupby(['subregion', 'type'])['capacity_mw'].sum()
    .div(1000)
    .rename('capacity_gw')
    .reset_index()
)

fig = px.bar(
    mix_by_subregion, x='subregion', y='capacity_gw', color='type',
    title='Operating capacity by subregion and technology (GW)',
    barmode='stack', color_discrete_map=TYPE_COLORS,
    category_orders={'type': TYPE_ORDER}
)
fig.update_xaxes(tickangle=45)
fig.show()

## 3. Historical generation trend — EI Statistical Review

Annual granularity gives trajectory context that the 6-snapshot IEA scenarios can't. Uses the `_twh` vars for `'Total World'` (the pre-flagged aggregate row). Note: `elect_twh` (world total electricity) only starts reporting in 1985 in this dataset, so the source-mix breakdown below is 1985-2024, not the full 1965-2024 span — years before 1985 have no valid total to derive the fossil residual from.

In [10]:
twh_vars = ['elect_twh', 'nuclear_twh', 'hydro_twh', 'solar_twh', 'wind_twh', 'biogeo_twh']

world_generation = (
    ei_stats[(ei_stats.country == 'Total World') & (ei_stats['var'].isin(twh_vars))]
    .pivot_table(index='year', columns='var', values='value')
    .rename(columns=lambda c: c.replace('_twh', ''))
)

# elect_twh (world total) only starts reporting in 1985 — before that, 'fossil'
# (derived as elect minus the other sources) is undefined, not zero. Drop those
# years rather than let a NaN denominator render as an empty/misleading band.
world_generation = world_generation.dropna(subset=['elect'])

world_generation['fossil'] = (
    world_generation['elect'] - world_generation[['nuclear', 'hydro', 'solar', 'wind', 'biogeo']].sum(axis=1)
)
world_generation.tail()

var,biogeo,elect,hydro,nuclear,solar,wind,fossil
year,,,,,,,
2020,688.732724,27016.62896,4359.128212,2689.004954,857.281356,1595.766601,16826.715113
2021,741.617309,28543.67601,4293.666067,2802.750489,1052.791021,1860.609167,17792.241957
2022,761.433470,29203.61600,4334.791986,2679.415444,1322.646522,2110.132834,17995.195744
2023,771.634722,29963.22999,4260.752877,2737.623183,1650.898587,2322.735330,18219.585291
2024,792.434159,31255.91438,4452.908891,2817.452321,2111.733333,2511.030709,18570.354967


In [11]:
mix_cols = ['nuclear', 'hydro', 'wind', 'solar', 'biogeo', 'fossil']
trend = world_generation[mix_cols].reset_index().melt(id_vars='year', var_name='source', value_name='twh')

fig = px.area(
    trend, x='year', y='twh', color='source',
    title=f'World electricity generation by source, {world_generation.index.min()}-{world_generation.index.max()} (TWh)'
)
fig.show()

In [12]:
nuclear_share = (world_generation['nuclear'] / world_generation['elect'] * 100).rename('nuclear_share_pct')

fig = px.line(
    nuclear_share.reset_index(), x='year', y='nuclear_share_pct',
    title=f"Nuclear's share of world electricity generation, {world_generation.index.min()}-{world_generation.index.max()} (%)"
)
fig.show()

## 4. GEM ↔ IEA region crosswalk check

GEM (`region`/`subregion`) and IEA (`region`) use different taxonomies with no clean automated join. This just lays both vocabularies side by side so the manual crosswalk used later (demand gap model) is grounded in what's actually present in the data.

In [13]:
print('GEM regions:      ', sorted(gipt.region.dropna().unique()))
print('GEM subregions:    ', sorted(gipt.subregion.dropna().unique()))
print()
print('IEA regions (regional file):', sorted(iea_regions.region.dropna().unique()))

GEM regions:       ['Africa', 'Americas', 'Asia', 'Europe', 'Oceania']
GEM subregions:     ['Australia and New Zealand', 'Central Asia', 'Eastern Asia', 'Eastern Europe', 'Latin America and the Caribbean', 'Melanesia', 'Micronesia', 'Northern Africa', 'Northern America', 'Northern Europe', 'Polynesia', 'South-eastern Asia', 'Southern Asia', 'Southern Europe', 'Sub-Saharan Africa', 'Western Asia', 'Western Europe']

IEA regions (regional file): ['Africa', 'Asia Pacific', 'Atlantic Basin', 'Brazil', 'Central and South America', 'China', 'East of Suez', 'Eurasia', 'Europe', 'European Union', 'India', 'Japan', 'Japan and Korea', 'Middle East', 'Non-OPEC plus', 'North America', 'OPECplus', 'Russia', 'Southeast Asia', 'United States']


Flags for the crosswalk (to define explicitly before the demand gap model):
- IEA `Africa` is continent-wide — GEM's `Sub-Saharan Africa` / `Northern Africa` subregions must be summed to compare, and the project's Sub-Saharan Africa focus has no matching IEA cut.
- IEA `Southeast Asia` aligns reasonably well with GEM's `South-eastern Asia` subregion.
- IEA offers single-country cuts (China, India, United States, Japan, Brazil, Russia) for granular comparison — GEM's `country` column supports the same joins directly.
- No 1:1 mapping exists for `Asia Pacific`, `Atlantic Basin`, `East of Suez`, `Non-OPEC plus`, `OPECplus`, `Eurasia` — these are IEA-specific groupings and would need bespoke country lists if used.

## 5. Nuclear pipeline status breakdown — GNPT

Preview for the Phase 3 nuclear pipeline module: operating capacity vs. pipeline (construction / pre-construction / announced) by country and status.

In [14]:
status_summary = (
    gnpt.groupby('status')
    .agg(units=('capacity_mw', 'size'), capacity_gw=('capacity_mw', lambda s: s.sum() / 1000))
    .sort_values('capacity_gw', ascending=False)
)
status_summary

,units,capacity_gw
status,,
cancelled,544,565.8205
operating,421,401.3410
announced,290,175.0000
retired,227,115.8958
pre-construction,140,107.0920
construction,76,81.5650
shelved,26,28.5120
mothballed,25,20.9270


In [15]:
pipeline_statuses = ['construction', 'pre-construction', 'announced']
pipeline_by_country = (
    gnpt[gnpt.status.isin(pipeline_statuses)]
    .groupby(['country', 'status'])['capacity_mw'].sum()
    .div(1000)
    .rename('capacity_gw')
    .reset_index()
)

top_countries = (
    pipeline_by_country.groupby('country')['capacity_gw'].sum()
    .sort_values(ascending=False).head(15).index
)

fig = px.bar(
    pipeline_by_country[pipeline_by_country.country.isin(top_countries)],
    x='country', y='capacity_gw', color='status',
    title='Nuclear pipeline capacity by country (top 15), GW',
    barmode='stack'
)
fig.update_xaxes(tickangle=45, categoryorder='total descending')
fig.show()

In [16]:
# Quick delivery-rate preview: operating vs. cancelled/shelved as a share of
# everything that's ever left "pipeline" status. Full historical delivery-rate
# derivation belongs in the Phase 3 nuclear pipeline module, not here.
resolved_statuses = ['operating', 'cancelled', 'shelved', 'retired']
resolved = gnpt[gnpt.status.isin(resolved_statuses)]
resolved.status.value_counts(normalize=True).mul(100).round(1).rename('share_pct')

status
cancelled    44.7
operating    34.6
retired      18.6
shelved       2.1
Name: share_pct, dtype: float64

## Next steps

This EDA feeds directly into Phase 3's three parallel modules:
- **Energy mix analysis** — extends Section 2 with historical share trends (Section 3) by region.
- **Demand gap model** — needs the crosswalk flagged in Section 4 resolved into an explicit mapping table before reconciling IEA projections with GEM bottom-up buildout.
- **Nuclear pipeline analysis** — extends Section 5's status breakdown into a full historical delivery-rate discount model using GNPT status-progression dates.